In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_PATH = Path("/Users/tradingworkspace/TradingWorkspace/data-engineering/Market Data/Cleaned/etf_prices_master.parquet")
OUTPUT_DIR = Path("/Users/tradingworkspace/TradingWorkspace/data-engineering/Market Data/Cleaned")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(DATA_PATH)
df = df.sort_values(["symbol", "date"]).reset_index(drop=True)

print(df.shape)
df.head()

(103131, 9)


,date,symbol,open,high,low,close,volume,nav,total_return_gross
0,2007-06-01,BIL,91.62,91.62,91.62,91.62,1450.0,91.56,91.62
1,2007-06-04,BIL,91.64,91.64,91.64,91.64,1050.0,91.60,91.64
2,2007-06-05,BIL,91.66,91.68,91.64,91.64,3750.0,91.61,91.64
3,2007-06-06,BIL,91.64,91.68,91.64,91.68,9750.0,91.63,91.68
4,2007-06-07,BIL,91.64,91.66,91.64,91.66,12150.0,91.64,91.66


In [2]:
ACTIVE_UNIVERSE = [
    "SPY", "QQQ",
    "VFH", "VHT", "VIS", "VPU",
    "VEA", "VWO", "INDA",
    "BIL", "SHY", "IEI", "TLT",
    "GLD", "SLV", "VNQ", "USO", "UNG"
]

EXCLUDED = ["TQQQ", "VIXY", "IBIT", "ETHA"]

df = df[df["symbol"].isin(ACTIVE_UNIVERSE)].copy()

print("Symbols in active universe:", sorted(df["symbol"].unique()))
print("Count:", df["symbol"].nunique())

Symbols in active universe: ['BIL', 'GLD', 'IEI', 'INDA', 'QQQ', 'SHY', 'SLV', 'SPY', 'TLT', 'UNG', 'USO', 'VEA', 'VFH', 'VHT', 'VIS', 'VNQ', 'VPU', 'VWO']
Count: 18


In [3]:
df["year_month"] = df["date"].dt.to_period("M")

rebalance_df = (
    df.groupby(["symbol", "year_month"], as_index=False)
      .tail(1)
      .copy()
)

rebalance_df = rebalance_df.sort_values(["date", "symbol"]).reset_index(drop=True)

print(rebalance_df.shape)
rebalance_df.head()

(4643, 10)


,date,symbol,open,high,low,close,volume,nav,total_return_gross,year_month
0,1993-02-26,SPY,44.44,44.44,44.19,44.41,66200.0,44.47,44.41,1993-02
1,1993-03-31,SPY,45.34,45.47,45.19,45.19,111600.0,45.19,45.40,1993-03
2,1993-04-30,SPY,44.13,44.28,44.03,44.03,88500.0,44.09,44.24,1993-04
3,1993-05-28,SPY,45.41,45.41,45.00,45.22,79100.0,45.25,45.43,1993-05
4,1993-06-30,SPY,45.13,45.22,45.00,45.06,437600.0,45.05,45.60,1993-06


In [4]:
bucket_map = {
    "SPY": "core_us",
    "QQQ": "core_us",

    "VFH": "us_sector",
    "VHT": "us_sector",
    "VIS": "us_sector",
    "VPU": "us_sector",

    "VEA": "international",
    "VWO": "international",
    "INDA": "international",

    "BIL": "defensive",
    "SHY": "defensive",
    "IEI": "defensive",
    "TLT": "defensive",

    "GLD": "real_assets",
    "SLV": "real_assets",
    "VNQ": "real_assets",
    "USO": "real_assets",
    "UNG": "real_assets",
}

rebalance_df["bucket"] = rebalance_df["symbol"].map(bucket_map)

missing_bucket = rebalance_df[rebalance_df["bucket"].isna()]["symbol"].unique()
print("Missing bucket assignments:", missing_bucket)

Missing bucket assignments: []


In [5]:
bucket_targets = {
    "core_us": 0.60,
    "us_sector": 0.20,
    "international": 0.10,
    "defensive": 0.05,
    "real_assets": 0.05,
}

bucket_weight_df = pd.DataFrame(
    list(bucket_targets.items()),
    columns=["bucket", "bucket_target_weight"]
)

bucket_weight_df

,bucket,bucket_target_weight
0,core_us,0.60
1,us_sector,0.20
2,international,0.10
3,defensive,0.05
4,real_assets,0.05


In [6]:
rebalance_df = rebalance_df.merge(bucket_weight_df, on="bucket", how="left")

bucket_counts = (
    rebalance_df.groupby(["date", "bucket"])["symbol"]
    .count()
    .rename("bucket_count")
    .reset_index()
)

rebalance_df = rebalance_df.merge(
    bucket_counts,
    on=["date", "bucket"],
    how="left"
)

rebalance_df["target_weight"] = (
    rebalance_df["bucket_target_weight"] / rebalance_df["bucket_count"]
)

rebalance_df["signal"] = True

rebalance_df.head(20)

,date,symbol,open,high,low,close,volume,nav,total_return_gross,year_month,bucket,bucket_target_weight,bucket_count,target_weight,signal
0,1993-02-26,SPY,44.44,44.44,44.19,44.41,66200.0,44.47,44.41,1993-02,core_us,0.6,1,0.6,True
1,1993-03-31,SPY,45.34,45.47,45.19,45.19,111600.0,45.19,45.40,1993-03,core_us,0.6,1,0.6,True
2,1993-04-30,SPY,44.13,44.28,44.03,44.03,88500.0,44.09,44.24,1993-04,core_us,0.6,1,0.6,True
3,1993-05-28,SPY,45.41,45.41,45.00,45.22,79100.0,45.25,45.43,1993-05,core_us,0.6,1,0.6,True
4,1993-06-30,SPY,45.13,45.22,45.00,45.06,437600.0,45.05,45.60,1993-06,core_us,0.6,1,0.6,True
5,1993-07-30,SPY,45.09,45.09,44.78,44.84,75300.0,44.86,45.38,1993-07,core_us,0.6,1,0.6,True
6,1993-08-31,SPY,46.41,46.56,46.34,46.56,66500.0,46.55,47.12,1993-08,core_us,0.6,1,0.6,True
7,1993-09-30,SPY,46.03,46.13,45.84,45.94,99300.0,45.89,46.78,1993-09,core_us,0.6,1,0.6,True
8,1993-10-29,SPY,46.81,46.88,46.78,46.84,80700.0,46.84,47.70,1993-10,core_us,0.6,1,0.6,True
9,1993-11-30,SPY,46.28,46.56,46.25,46.34,230000.0,46.39,47.19,1993-11,core_us,0.6,1,0.6,True


In [7]:
# Sum of weights by date
weight_check = rebalance_df.groupby("date")["target_weight"].sum()
print(weight_check.tail(10))

# Sum of weights by bucket for latest date
latest_date = rebalance_df["date"].max()
latest_slice = rebalance_df[rebalance_df["date"] == latest_date].copy()

latest_slice.groupby("bucket")["target_weight"].sum().sort_index()

date
2025-06-30    1.0
2025-07-31    1.0
2025-08-29    1.0
2025-09-30    1.0
2025-10-31    1.0
2025-11-28    1.0
2025-12-31    1.0
2026-01-30    1.0
2026-02-27    1.0
2026-03-06    1.0
Name: target_weight, dtype: float64


bucket
core_us          0.60
defensive        0.05
international    0.10
real_assets      0.05
us_sector        0.20
Name: target_weight, dtype: float64

In [8]:
latest_slice[[
    "date", "symbol", "bucket", "close", "target_weight"
]].sort_values(["bucket", "symbol"])

,date,symbol,bucket,close,target_weight
4629,2026-03-06,QQQ,core_us,599.75,0.300000
4632,2026-03-06,SPY,core_us,672.38,0.300000
4625,2026-03-06,BIL,defensive,91.44,0.012500
4627,2026-03-06,IEI,defensive,119.41,0.012500
4630,2026-03-06,SHY,defensive,82.73,0.012500
4633,2026-03-06,TLT,defensive,88.46,0.012500
4628,2026-03-06,INDA,international,49.99,0.033333
4636,2026-03-06,VEA,international,65.28,0.033333
4642,2026-03-06,VWO,international,54.47,0.033333
4626,2026-03-06,GLD,real_assets,473.51,0.010000


In [9]:
targets = rebalance_df[[
    "date",
    "symbol",
    "bucket",
    "close",
    "volume",
    "target_weight",
    "signal",
]].copy()

targets = targets.sort_values(["date", "symbol"]).reset_index(drop=True)

print(targets.shape)
targets.head(20)

(4643, 7)


,date,symbol,bucket,close,volume,target_weight,signal
0,1993-02-26,SPY,core_us,44.41,66200.0,0.6,True
1,1993-03-31,SPY,core_us,45.19,111600.0,0.6,True
2,1993-04-30,SPY,core_us,44.03,88500.0,0.6,True
3,1993-05-28,SPY,core_us,45.22,79100.0,0.6,True
4,1993-06-30,SPY,core_us,45.06,437600.0,0.6,True
5,1993-07-30,SPY,core_us,44.84,75300.0,0.6,True
6,1993-08-31,SPY,core_us,46.56,66500.0,0.6,True
7,1993-09-30,SPY,core_us,45.94,99300.0,0.6,True
8,1993-10-29,SPY,core_us,46.84,80700.0,0.6,True
9,1993-11-30,SPY,core_us,46.34,230000.0,0.6,True


In [10]:
dup_count = targets.duplicated(subset=["date", "symbol"]).sum()
print("Duplicate date-symbol rows:", dup_count)

Duplicate date-symbol rows: 0


In [11]:
targets_csv = OUTPUT_DIR / "etf_targets_monthly.csv"
targets_parquet = OUTPUT_DIR / "etf_targets_monthly.parquet"

targets.to_csv(targets_csv, index=False)
targets.to_parquet(targets_parquet, index=False)

print("Saved:")
print(targets_csv)
print(targets_parquet)

Saved:
/Users/tradingworkspace/TradingWorkspace/data-engineering/Market Data/Cleaned/etf_targets_monthly.csv
/Users/tradingworkspace/TradingWorkspace/data-engineering/Market Data/Cleaned/etf_targets_monthly.parquet
